# Lab: Shortest Paths with Bellman-Ford and Floyd-Warshall

**Topics:** Bellman-Ford · Floyd-Warshall
**Time:** about 70–90 minutes  **Language:** Python 3.9+ (no extra libraries needed)

## What you will do

| Task | Topic | What you implement | Points |
|------|-------|--------------------|--------|
| 1 | Bellman-Ford | `bellman_ford`, `build_path`, `bellman_ford_early` | 45 |
| 2 | Floyd-Warshall | `floyd_warshall`, `fw_path`, `all_pairs_bf` (cross-check with Task 1) | 45 |
| – | Reflection | written answers at the end | 10 |
| | | **Total** | **100** |

## How this notebook works

1. Read the instruction text above each code cell.
2. Complete every cell marked `# TODO`. Keep the function names and signatures unchanged.
3. Run the **test cell** under each part. It prints ✅ or ❌ for every check and records your points.
4. At the end, run the **score summary** cell, answer the reflection questions, then save and submit.

> The functions start as empty stubs (`return None`), so **Run All** will not crash. You will simply see ❌ until you fill them in.

**Rules:** write your own code; you may use the lecture slides and the Python documentation. Do not change the test cells.

## 0. Setup (run this cell first)

This cell defines `INF`, the example graphs and the small testing helper `check(...)`. You do not need to edit it.

In [ ]:
import random, time

INF = float("inf")

# ---- tiny testing helper (do not edit) ------------------------------------
RESULTS = {}

def check(task, name, test, pts=1):
    """Run test() and record whether it returned True. Exceptions count as a fail."""
    try:
        ok = bool(test())
        err = ""
    except Exception as e:
        ok, err = False, f"  ({type(e).__name__}: {e})"
    RESULTS.setdefault(task, []).append((name, ok, pts))
    print(("✅" if ok else "❌"), f"[{task}] {name} ({pts} pt){err}")

def summary():
    print(f"{'Task':<22}{'Points':>8}{'Max':>6}")
    total = mx = 0
    for task, rows in RESULTS.items():
        got = sum(p for _, ok, p in rows if ok)
        top = sum(p for _, _, p in rows)
        total += got; mx += top
        print(f"{task:<22}{got:>8}{top:>6}")
    print("-" * 36)
    print(f"{'AUTOMATIC TOTAL':<22}{total:>8}{mx:>6}")

# ---- graphs used in the lab (do not edit) ----------------------------------
V = ["S", "A", "B", "C", "D"]
E = [("S", "A", 6), ("S", "B", 3), ("B", "A", -2),
     ("A", "C", 3), ("B", "D", 7), ("C", "D", -1)]          # has a negative edge, no negative cycle

V_CYCLE = ["S", "A", "B"]
E_CYCLE = [("S", "A", 2), ("A", "B", 2), ("B", "S", -5)]    # cycle S->A->B->S costs -1

def random_graph(n, m, seed=0):
    """Random directed graph with some negative edges but NO negative cycle."""
    rnd = random.Random(seed)
    pot = [rnd.randint(0, 10) for _ in range(n)]
    edges = {}
    while len(edges) < m:
        u, v = rnd.sample(range(n), 2)
        edges[(u, v)] = rnd.randint(1, 9) + pot[u] - pot[v]
    return [(u, v, w) for (u, v), w in edges.items()]

print("Setup done.")

---
# Task 1 · Bellman-Ford (45 points)

## Background

Bellman-Ford finds the cheapest path from one **source** to every vertex, even when some edges have **negative** weights.

It works in rounds. After round $k$, `dist[b]` is the cost of the cheapest path from the source to `b` that uses **at most $k$ edges**:

$$d^{(k)}[b] = \min\Big\{\, d^{(k-1)}[b],\ \min_{(a,b)\in E}\big( d^{(k-1)}[a] + w(a,b) \big) \Big\}$$

* A simple path has at most $n-1$ edges, so $n-1$ rounds are enough when there is no negative cycle.
* If an edge can **still** be improved after $n-1$ rounds, a negative cycle is reachable from the source.
* **Key rule:** in round $k$ read only the *old* values ($k-1$) and write the *new* ones.

### Our graph `V`, `E` (source `S`)

| From | To | Weight |
|------|----|--------|
| S | A | 6 |
| S | B | 3 |
| B | A | −2 |
| A | C | 3 |
| B | D | 7 |
| C | D | −1 |

## Part 1A · Trace by hand (no code)

Before coding, fill in this table on paper or in the cell below (use `inf` for ∞). Each row is one round.

| round | S | A | B | C | D |
|-------|---|---|---|---|---|
| d(0)  | 0 | inf | inf | inf | inf |
| d(1)  | ? | ? | ? | ? | ? |
| d(2)  | ? | ? | ? | ? | ? |
| d(3)  | ? | ? | ? | ? | ? |
| d(4)  | ? | ? | ? | ? | ? |

Write your answers here (a Python list per round is fine):

In [ ]:
my_trace = {
    1: [0, 6, 3, INF, INF],
    2: [0, 1, 3, 9, 10],
    3: [0, 1, 3, 4, 8],
    4: [0, 1, 3, 4, 3],
}

## Part 1B · Implement `bellman_ford`

**Instructions**

1. Create `dist` (all `INF`, source `0`) and `prev` (all `None`).
2. Create `history = [dist.copy()]`; it stores $d^{(0)}, d^{(1)}, \dots$ for printing and testing.
3. Repeat `len(vertices) - 1` rounds. In each round:
   * `new = dist.copy()` (this keeps the "fewer edges" case),
   * for every edge `(a, b, w)`: if `dist[a] + w < new[b]`, set `new[b]` and `prev[b] = a`,
   * after the edge loop set `dist = new` and append `dist.copy()` to `history`.
4. After all rounds, loop over the edges once more. If any edge still improves (`dist[a] + w < dist[b]`), `raise ValueError("negative cycle reachable from source")`.
5. `return dist, prev, history`.

**Hint:** compare against `dist[a]` (old values) but write into `new[b]`. If you write into `dist` directly you have changed the algorithm (it would still often work, but your `history` rows will not match the tests).

In [ ]:
def bellman_ford(vertices, edges, source):
    """Return (dist, prev, history). Raise ValueError on a reachable negative cycle."""
    dist = {}
    prev = {}
    for vertex in vertices:
        dist[vertex] = INF
        prev[vertex] = None
    dist[source] = 0
    history = [dist.copy()]

    for _ in range(len(vertices) - 1):
        new = dist.copy()
        for a, b, w in edges:
            if dist[a] + w < new[b]:
                new[b] = dist[a] + w
                prev[b] = a
        dist = new
        history.append(dist.copy())

    for a, b, w in edges:
        if dist[a] + w < dist[b]:
            raise ValueError("negative cycle reachable from source")

    return dist, prev, history

## Part 1C · Rebuild a path

`prev[v]` stores the vertex just before `v` on the best path. Starting at the target, follow `prev` back to the source and reverse the list.

Implement `build_path(prev, target)` so that `build_path(prev, "D")` returns `["S", "B", "A", "C", "D"]`.

In [ ]:
def build_path(prev, target):
    """Return the list of vertices from the source to target."""
    path = []
    while target is not None:
        path.append(target)
        target = prev[target]
    path.reverse()
    return path

## Part 1D · Early exit (challenge)

If a round changes nothing, later rounds will not change anything either, so we can stop early.

Write `bellman_ford_early(vertices, edges, source)` that behaves like `bellman_ford` but:

* stops **right after the first round in which no distance changed**,
* returns `(dist, prev, rounds)` where `rounds` is the number of rounds that were executed (including the final no-change round),
* still raises `ValueError` for a negative cycle (do the extra check pass if it never stopped early).

*Example:* on a 6-vertex chain `S→A→B` (other vertices unreachable) round 3 changes nothing, so `rounds == 3` even though $n-1 = 5$.

In [ ]:
def bellman_ford_early(vertices, edges, source):
    """Return (dist, prev, rounds). Stop after the first round with no change."""
    dist = {}
    prev = {}
    for vertex in vertices:
        dist[vertex] = INF
        prev[vertex] = None
    dist[source] = 0
    rounds = 0

    for _ in range(len(vertices) - 1):
        new = dist.copy()
        changed = False
        for a, b, w in edges:
            if dist[a] + w < new[b]:
                new[b] = dist[a] + w
                prev[b] = a
                changed = True
        dist = new
        rounds += 1
        if not changed:
            return dist, prev, rounds

    for a, b, w in edges:
        if dist[a] + w < dist[b]:
            raise ValueError("negative cycle reachable from source")

    return dist, prev, rounds

### Test Task 1 (run this cell)

In [ ]:
T = "Task 1"
expected_hist = [
    [0, INF, INF, INF, INF], [0, 6, 3, INF, INF], [0, 1, 3, 9, 10],
    [0, 1, 3, 4, 8], [0, 1, 3, 4, 3],
]
check(T, "my_trace matches d(1)…d(4)", lambda: [my_trace[k] for k in (1, 2, 3, 4)] == expected_hist[1:], 6)
check(T, "final distances are correct",
      lambda: bellman_ford(V, E, "S")[0] == {"S": 0, "A": 1, "B": 3, "C": 4, "D": 3}, 9)
check(T, "history rows d(0)…d(4) are correct",
      lambda: [[h[v] for v in V] for h in bellman_ford(V, E, "S")[2]] == expected_hist, 7)
check(T, "build_path(prev, 'D') == S,B,A,C,D",
      lambda: build_path(bellman_ford(V, E, "S")[1], "D") == ["S", "B", "A", "C", "D"], 6)
check(T, "build_path(prev, 'S') == ['S']",
      lambda: build_path(bellman_ford(V, E, "S")[1], "S") == ["S"], 2)

def _raises(fn):
    try:
        fn()
    except ValueError:
        return True
    return False

check(T, "negative cycle raises ValueError", lambda: _raises(lambda: bellman_ford(V_CYCLE, E_CYCLE, "S")), 6)

V6 = ["S", "A", "B", "C", "D", "E"]
E6 = [("S", "A", 1), ("A", "B", 1)]
check(T, "early exit: same distances", lambda: bellman_ford_early(V, E, "S")[0] == bellman_ford(V, E, "S")[0], 3)
check(T, "early exit: stops after 3 rounds on the chain graph", lambda: bellman_ford_early(V6, E6, "S")[2] == 3, 4)
check(T, "early exit: still detects negative cycle", lambda: _raises(lambda: bellman_ford_early(V_CYCLE, E_CYCLE, "S")), 2)

---
# Task 2 · Floyd-Warshall (45 points)

## Background

Floyd-Warshall computes the shortest distance between **every pair** of vertices. Number the vertices `0 … n-1`. Let `D[i][j]` hold the best known distance from `i` to `j`. We allow one more vertex `k` as a **middle stop** each round:

$$D^{(k)}[i][j] = \min\{\, D^{(k-1)}[i][j],\ D^{(k-1)}[i][k] + D^{(k-1)}[k][j] \,\}$$

* Start: `D[i][i] = 0`, `D[i][j] = weight` if the edge exists, otherwise `INF`.
* Three nested loops (`k` outermost) → $O(n^3)$ time.
* If some `D[v][v] < 0` at the end, there is a negative cycle.
* To rebuild paths keep `nxt[i][j]` = the **next vertex after `i`** on the best path to `j`.

### Graph for this task (vertices 0–3, 0-based)

`0→1 (3)`, `0→2 (5)`, `1→2 (−2)`, `2→3 (2)`, `3→0 (4)`

## Part 2A · Implement `floyd_warshall`

1. Create `D` (n×n of `INF`) and `nxt` (n×n of `None`). Set `D[i][i] = 0`.
2. For every edge `(u, v, w)` set `D[u][v] = w` and `nxt[u][v] = v`.
3. Loop `k`, then `i`, then `j`. If `D[i][k] + D[k][j] < D[i][j]`, update `D[i][j]` and set `nxt[i][j] = nxt[i][k]`.
4. If any diagonal entry is negative, `raise ValueError("negative cycle detected")`.
5. `return D, nxt`.

*Note:* updating `D` in place is safe here because row `k` and column `k` do not change during round `k` when there is no negative cycle.

In [ ]:
def floyd_warshall(n, edges):
    """Return (D, nxt). Raise ValueError on a negative cycle."""
    # TODO
    return None

## Part 2B · Rebuild a path

Implement `fw_path(nxt, i, j)`:

* if `nxt[i][j]` is `None` there is no path: return `[]`,
* otherwise start with `path = [i]` and, while `i != j`, move `i = nxt[i][j]` and append it.

For the graph above the path from `3` to `2` is `[3, 0, 1, 2]` (cost 4 + 3 − 2 = 5).

In [ ]:
def fw_path(nxt, i, j):
    """Return the vertex list from i to j, or [] if j is unreachable."""
    # TODO
    return None

## Part 2C · Cross-check with Bellman-Ford (Task 1)

Two different algorithms must agree. Running Bellman-Ford from **every** vertex gives the same all-pairs distances as Floyd-Warshall. Complete `all_pairs_bf(n, edges)` using your `bellman_ford` (use `list(range(n))` as the vertex list) and return an n×n list of lists.

In [ ]:
def all_pairs_bf(n, edges):
    """Return an n x n matrix by running bellman_ford from every source."""
    # TODO
    return None

### Test Task 2 (run this cell)

In [ ]:
T = "Task 2"
E4 = [(0, 1, 3), (0, 2, 5), (1, 2, -2), (2, 3, 2), (3, 0, 4)]
expected_D = [[0, 3, 1, 3], [4, 0, -2, 0], [6, 9, 0, 2], [4, 7, 5, 0]]
check(T, "distance matrix is correct", lambda: floyd_warshall(4, E4)[0] == expected_D, 15)
check(T, "fw_path(3 -> 2) == [3, 0, 1, 2]", lambda: fw_path(floyd_warshall(4, E4)[1], 3, 2) == [3, 0, 1, 2], 7)
check(T, "no path -> []", lambda: fw_path(floyd_warshall(3, [(0, 1, 1)])[1], 1, 0) == [], 4)
check(T, "unreachable distance is INF", lambda: floyd_warshall(3, [(0, 1, 1)])[0][1][0] == INF, 2)
check(T, "negative cycle raises ValueError",
      lambda: _raises(lambda: floyd_warshall(3, [(0, 1, 2), (1, 2, 2), (2, 0, -5)])), 6)
RG = random_graph(12, 40, seed=7)
check(T, "matches Bellman-Ford from every vertex (n=12)",
      lambda: floyd_warshall(12, RG)[0] == all_pairs_bf(12, RG), 7)
check(T, "matches Bellman-Ford on a second random graph (n=15)",
      lambda: floyd_warshall(15, random_graph(15, 60, seed=3))[0] == all_pairs_bf(15, random_graph(15, 60, seed=3)), 4)

## Part 2D · Compare running times (no coding)

Run the cell below. It times your Bellman-Ford-from-every-vertex against your Floyd-Warshall on a larger graph. Use the numbers in the reflection questions.

In [ ]:
n, m = 60, 500
G = random_graph(n, m, seed=11)
try:
    t0 = time.time(); a = all_pairs_bf(n, G);  t_bf = time.time() - t0
    t0 = time.time(); b, _ = floyd_warshall(n, G); t_fw = time.time() - t0
    print(f"n={n}, m={m}")
    print(f"n x Bellman-Ford : {t_bf:.3f} s   (about n^2*m = {n*n*m:,} edge relaxations)")
    print(f"Floyd-Warshall   : {t_fw:.3f} s   (about n^3   = {n**3:,} steps)")
    print("same answer:", a == b)
except Exception as e:
    print("Finish Tasks 1 and 3 first, then re-run this cell.", type(e).__name__)

---
# Reflection (10 points, written)

Answer in 2–4 sentences each by editing the cells below. (Graded by hand.)

1. **(Task 1)** Why are exactly $n-1$ rounds enough when there is no negative cycle? What does it mean if an edge still improves in round $n$?
2. **(Task 1)** Why must round $k$ read the *old* values and write into a copy (`new`)?
3. **(Task 2)** Why is Floyd-Warshall $O(n^3)$? When would you prefer it to running Dijkstra or Bellman-Ford from every vertex?
4. **(Task 2)** Using your timing from Part 2D, which was faster and does the result match the theory?

**Answer 1:** Without a negative cycle, a shortest path does not need to repeat vertices, so it has at most n-1 edges. If a distance still improves in round n, a negative cycle is reachable from the source.

**Answer 2:** Using the old values makes each round add at most one edge to a path. If we update dist directly, a new value can be used again in the same round, so the trace can depend on the edge order.

**Answer 3:**

**Answer 4:**

## Final score summary and submission

Run the cell below. Make sure your reflection answers are filled in, then **Kernel → Restart & Run All**, check that no cell shows an error, save the notebook and submit the `.ipynb` file.

In [ ]:
summary()
print()
print("Manual part: reflection answers = 10 points (graded by the instructor).")

### Submission checklist

- [ ] All `# TODO` cells completed; no function renamed
- [ ] `Restart & Run All` finishes without errors
- [ ] Every test prints ✅ (or you can explain each ❌)
- [ ] Hand-trace table in Part 1A filled in
- [ ] Reflection answers written
- [ ] File saved as `lab_<your_name>.ipynb`